<a href="https://colab.research.google.com/github/luisy-eng/ECE491E_Lab1/blob/main/Task3_CustomModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3: Customize Model — Deeper Feedforward Neural Network

The purpose of Task 3 is to replace the neural network architecture used in
the PyTorch tutorial with a deeper feedforward neural network designed for the
CIFAR-10 input prepared in Task 2.

The required architecture consists of:

- An input layer matching the CIFAR-10 feature size.
- A first hidden layer containing 128 neurons.
- ReLU activation.
- Dropout with a probability of 0.3.
- A second hidden layer containing 64 neurons.
- ReLU activation.
- Dropout with a probability of 0.3.
- A single output neuron for regression.

The model will be constructed and verified in this notebook. The regression
loss function will be introduced separately in Task 4.


## 1. Import Required Libraries

PyTorch's `torch` package provides the tensor operations required by the model,
while `torch.nn` contains the neural network layers and activation functions
used to construct the architecture.

The model will also be configured to use a GPU through CUDA when one is
available. Otherwise, it will run on the CPU.

In [1]:
import torch
from torch import nn

## 2. Select the Computing Device

Neural network operations can be accelerated using a GPU. PyTorch provides
CUDA support for compatible NVIDIA GPUs.

The following code checks whether CUDA is available. If a GPU is available,
the model will use it; otherwise, the CPU will be used.

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Using {device} device")

Using cuda device


## 3. Determine the Input Feature Size

CIFAR-10 images have three RGB color channels and dimensions of 32 × 32
pixels. Unlike a convolutional neural network, the feedforward network used
for this task requires each image to be converted into a one-dimensional
feature vector.

The number of input features is therefore:

\[
3 \times 32 \times 32 = 3072
\]

The model will use a `Flatten` layer to automatically convert an input tensor
with dimensions `[3, 32, 32]` into a vector containing 3,072 features.

In [3]:
channels = 3
height = 32
width = 32

input_features = channels * height * width

print(f"Input features: {input_features}")

Input features: 3072


## 4. Define the Modified Neural Network

The modified model is implemented as a subclass of PyTorch's `nn.Module`.

The network begins with `nn.Flatten()`, which converts each CIFAR-10 image
from `[3, 32, 32]` into a vector of 3,072 input features.

The first fully connected layer maps the 3,072 input features to 128 hidden
neurons. A ReLU activation function is then applied, followed by a dropout
layer with a probability of 0.3.

The second fully connected layer reduces the 128 features to 64 hidden
neurons. This is followed by another ReLU activation and another dropout layer
with a probability of 0.3.

Finally, the output layer maps the 64 hidden features to a single output
neuron, as required for the regression model.

The resulting architecture is:

`3072 → 128 → ReLU → Dropout(0.3) → 64 → ReLU → Dropout(0.3) → 1`

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.network = nn.Sequential(
            nn.Linear(3072, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)


model = NeuralNetwork().to(device)

print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=3072, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 5. ReLU Activation Function

The Rectified Linear Unit (ReLU) activation function is applied after each
hidden linear layer.

ReLU is defined as:

\[
f(x) = \max(0,x)
\]

This means that negative input values are replaced with zero while positive
values are preserved.

Without nonlinear activation functions, multiple linear layers would still
behave as a single linear transformation. ReLU introduces nonlinearity,
allowing the network to learn more complex relationships between the input
features and output.

## 6. Dropout Regularization

A dropout layer with a probability of 0.3 is placed after each hidden layer.

During training, dropout randomly sets a portion of the layer's activations to
zero. For a dropout probability of:

\[
p = 0.3
\]

approximately 30% of the activations are dropped during each training pass.

Dropout acts as a regularization technique and can help reduce overfitting by
preventing the network from relying too heavily on individual neurons.

Dropout behaves differently during training and evaluation. During model
evaluation, dropout is disabled so that all learned activations can be used.

## 7. Verify the Model Output

A synthetic batch of CIFAR-10-shaped tensors is passed through the network to
verify that the architecture accepts the expected input dimensions and
produces the required output dimensions.

A batch containing 64 CIFAR-10 images has the shape:

`[64, 3, 32, 32]`

After flattening, each image contains 3,072 features. Because the final layer
contains one neuron, the expected model output has the shape:

`[64, 1]`

In [5]:
test_input = torch.randn(64, 3, 32, 32).to(device)

model.eval()

with torch.no_grad():
    test_output = model(test_input)

print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")

Input shape:  torch.Size([64, 3, 32, 32])
Output shape: torch.Size([64, 1])


## 8. Model Parameters

The number of trainable parameters in the modified neural network is calculated
to provide additional information about the size of the model.

Each fully connected layer contains trainable weights and biases, while the
ReLU, Dropout, and Flatten layers do not contain trainable parameters.

In [6]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")

Total parameters: 401,665
Trainable parameters: 401,665


## 9. Task 3 Summary

Task 3 replaced the neural network architecture used in the original PyTorch
tutorial with the deeper feedforward architecture specified by the project.

The completed model uses:

- 3,072 input features obtained from each flattened CIFAR-10 image.
- A first hidden layer containing 128 neurons.
- ReLU activation.
- Dropout with a probability of 0.3.
- A second hidden layer containing 64 neurons.
- ReLU activation.
- A second dropout layer with a probability of 0.3.
- One output neuron for regression.

The final architecture is:

`3072 → 128 → ReLU → Dropout(0.3) → 64 → ReLU → Dropout(0.3) → 1`

The model contains 401,665 trainable parameters.

A test batch with dimensions `[64, 3, 32, 32]` was successfully passed through
the network and produced an output with dimensions `[64, 1]`, confirming that
the architecture operates with the expected CIFAR-10 input and regression
output dimensions.

The regression loss function and model training procedure will be implemented
in Task 4.